# 뉴스 추출 결과 정리: 최종본 만들기

`news_extractor.ipynb`가 기사마다 따로 받아 둔 LLM 결과(Codex·배치)를 **하나의 파일로 합치고, 규칙으로 고쳐 최종본을 만든다.** `dart_business_cleaner.ipynb`와 같은 방식이다(`graphDB.md` 5.3, 6장).

- 원본 결과(`extract/raw/`)는 건드리지 않는다. 최종본에서 고친 것은 모두 변경 기록(`changes.csv`)에 남긴다.
- 서버나 LLM에 요청하지 않는다. 몇 번을 다시 실행해도 된다.
- **최종 지시문 v5의 결과만 쓴다.** 같은 기사에 결과가 여럿이면 배치(API) 결과를, 그 안에서는 마지막 실행을 쓴다. 모든 기사에 v5 결과가 있어야 한다(없으면 멈춘다).

**고치는 것**

| 번호 | 내용 | 처리 |
|---|---|---|
| ① | 근거가 원문과 다름 | 원문에서 가장 비슷한 부분으로 바꾼다. 비슷한 부분이 없으면 그 근거와 그 근거를 쓰는 관계·사건을 뺀다 |
| ② | 관계로 보지 않는 것 | 자기 자신이 상대인 관계, 증권사 투자의견을 적은 관계(지시문이 막지만 남은 경우), 없는 개체를 가리키는 관계, 없는 근거 번호를 가리키는 관계(사건은 없는 번호만 빼고, 남는 근거가 없으면 사건을 뺀다) |
| ③ | 겹치는 관계 | 같은 주체·역할·대상·제품의 관계는 하나만. 공급(A → B, 제품 P)이 있으면 같은 거래인 B의 구매·사용(B → P 또는 B → A)은 뺀다 |
| ④ | 너무 일반적인 검색어 | "AI", "oil" 같은 말은 개체의 검색어에서 뺀다(개체는 남긴다) |

**입력**

| 파일 | 내용 |
|---|---|
| `source/news/extract/runs.csv`, `raw/` | 실행 기록과 원본 결과 |
| `source/news/filter/article_filter.csv` | 기사별 사용 여부 |
| `source/news/snapshot_*/news.sqlite3` | 기사 원문(근거 확인용, 가장 최근 스냅샷) |

**출력** (`source/news/final/`)

| 파일 | 내용 |
|---|---|
| `news_final.json` | 최종본. 기사마다 메타 정보(매체, 제목, 주소, 발행 시각, 지시문 버전)와 정리한 결과 |
| `changes.csv` | 고친 것 한 줄: 기사, 번호, 항목, 바꾸기 전, 바꾼 뒤, 이유 |

**걸리는 시간**: 약 1분.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `news_extractor` → **`news_cleaner`** → `name_unifier` |
| 외부 요청·비용 | 없음 |
| 다시 실행 | 언제든 된다. 결과 파일은 내용이 바뀔 때만 다시 쓴다 |
| 멈추는 경우 | v5 결과가 없는 기사, 결과를 만들 때와 입력이 달라진 기사, 마지막 점검에서 어긋난 것이 있을 때 |

## 1. 설정

In [1]:
# ── 설정: 폴더, 지시문 버전, 결과 고르는 순서, 관계 역할 목록 ──
import difflib
import hashlib
import html
import json
import re
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
NEWS = ROOT / 'source' / 'news'
EXTRACT = NEWS / 'extract'
FINAL = NEWS / 'final'
FINAL.mkdir(parents=True, exist_ok=True)

MODEL = 'gpt-6.1-terra'
PROMPT_VERSION = 'v5'            # 최종 지시문
RUNNERS = ['batch', 'codex']     # 같은 버전이면 배치(API) 결과를 먼저 쓴다. 매일 밤 처리와 같은 수단으로 맞추기 위해서다
ROLES_V5 = ['공급', '구매·사용', '생산', '수주·발주', '인수·투자', '매각', '합작·제휴', '라이선스', '경쟁', '규제·인허가',
            '시설 건설·운영', '영향', '소송·분쟁', '자금 지원', '정책 입장', '기타']
# 근거 비교용: 공백을 모두 지운 글자열
squash = lambda s: re.sub(r'\s+', '', s or '')
# 기사 원문은 가장 최근 스냅샷에서 읽는다(추출 때 쓴 것과 같아야 한다)
SNAPSHOT = sorted(NEWS.glob('snapshot_*/news.sqlite3'))[-1]
print('기사 원문', SNAPSHOT.relative_to(ROOT))

기사 원문 source/news/snapshot_20261112_1127/news.sqlite3


## 2. 결과 모으기

- `runs.csv`에서 모델이 `gpt-6.1-terra`이고 성공한 실행만 고른다. 기사마다 `VERSIONS` 순서로 가장 새 버전을 쓰고, 같은 버전 안에서는 `RUNNERS` 순서(배치 먼저)로, 같은 수단 안에서는 마지막 실행을 쓴다.
- 결과를 만들 때의 입력 글과 지금 입력 글이 같은지(`input_hash`) 확인한다. 입력 글은 `news_extractor`와 똑같이 만든다(제목·매체·발행 시각 + 본문 또는 요약).

In [2]:
# ── 결과 모으기: 기사마다 쓸 결과 하나를 고르고, 입력이 그대로인지 확인한다 ──
def strip_html(s):
    """RSS 요약의 HTML 태그를 지우고 글자만 남긴다."""
    s = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', s or '', flags=re.S | re.I)
    s = html.unescape(re.sub(r'<[^>]+>', ' ', s))
    return re.sub(r'[ \t]+', ' ', s).strip()


def parse_result(resp):
    """원본 응답에서 결과 JSON을 꺼낸다(Codex: 최종 메시지, 배치 한 줄: 응답 본문의 output)."""
    if 'custom_id' in resp:
        resp = (resp.get('response') or {}).get('body') or {}
    if isinstance(resp.get('output'), list):
        text = next((c.get('text') or '' for item in resp['output'] if item.get('type') == 'message'
                     for c in item.get('content') or [] if c.get('type') == 'output_text'), '')
    else:
        text = resp.get('last_message') or ''
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    return json.loads(text[text.find('{'):])


# 거르기 결과에서 본문이나 요약을 쓰는 기사만
flt = pd.read_csv(NEWS / 'filter' / 'article_filter.csv', dtype=str).fillna('')
flt = flt[flt['use'].isin(['body', 'summary'])]
with sqlite3.connect(SNAPSHOT) as con:
    art = pd.read_sql('select article_id, source, title, url, published_utc, summary_html, body_text from articles', con)
art = flt[['article_id', 'use']].merge(art, on='article_id').set_index('article_id')


def input_text(r):
    """추출할 때 LLM에 보낸 입력 글을 똑같이 다시 만든다(입력 해시 비교용)."""
    body = r['body_text'] if r['use'] == 'body' else strip_html(r['summary_html'])
    return f"Title: {r['title']}\nSource: {r['source']}\nPublished: {r['published_utc']}\n\n{(body or '').strip()}"


TEXT = {a: input_text(r) for a, r in art.iterrows()}

runs = pd.read_csv(EXTRACT / 'runs.csv', dtype=str)
# 성공한 v5 실행만 고르고, 기사마다 배치 결과 → 최근 실행 순으로 하나만 남긴다
ok = runs[(runs['status'] == 'ok') & (runs['model'] == MODEL) & (runs['prompt_version'] == PROMPT_VERSION)].copy()
ok['rrank'] = ok['runner'].map({v: i for i, v in enumerate(RUNNERS)}).fillna(len(RUNNERS))
ok = ok.sort_values(['rrank', 'run_utc'], ascending=[True, False]).drop_duplicates('article_id').set_index('article_id')

# 결과가 없거나, 결과를 만들 때와 지금 입력이 다르면 멈춘다(다시 추출해야 한다)
missing = [a for a in TEXT if a not in ok.index]
stale = [a for a in TEXT if a in ok.index and ok.at[a, 'input_hash'] != hashlib.sha256(TEXT[a].encode('utf-8')).hexdigest()[:16]]
assert not missing and not stale, f'결과 없음 {len(missing)}건, 입력 바뀜 {len(stale)}건'

# 원본 응답 파일을 읽어 결과 JSON만 꺼낸다
results, meta = {}, {}
for a in TEXT:
    r = ok.loc[a]
    folder = ('api-' if r['runner'] == 'batch' else 'codex-') + MODEL
    results[a] = parse_result(json.loads((EXTRACT / 'raw' / r['prompt_version'] / folder / r['raw_file']).read_text(encoding='utf-8')))
    meta[a] = dict(prompt_version=r['prompt_version'], runner=r['runner'], raw_file=r['raw_file'])

# 정리 전 항목 수(맨 아래 비교 표에 쓴다)
raw_counts = {a: {k: len(o[k]) for k in ('entities', 'relations', 'events', 'evidence')} for a, o in results.items()}
print('기사', len(results), '/ 지시문 버전', pd.Series([m['prompt_version'] for m in meta.values()]).value_counts().to_dict(),
      '/ 실행 수단', pd.Series([m['runner'] for m in meta.values()]).value_counts().to_dict())

기사 2471 / 지시문 버전 {'v5': 2471} / 실행 수단 {'batch': 2471}


## 3. 변경 기록 준비

In [3]:
# ── 변경 기록: 고칠 때마다 한 줄씩 남긴다 ──
changes = []


def log(aid, rule, section, before, after, why):
    """고친 내용 한 줄을 변경 기록에 더한다."""
    changes.append(dict(article_id=aid, title=art.at[aid, 'title'][:80], rule=rule, section=section,
                        before=str(before)[:300], after=str(after)[:300], why=why))


def rel_label(o, x):
    """관계를 "주체 -역할-> 대상" 글로 만든다(기록·확인용)."""
    names = {e['id']: e['name_en'] or e['name'] for e in o['entities']}
    return f"{names.get(x['subject'], x['subject'])} -{x['role']}-> {names.get(x['object'], x['object'])}"

## 4. 역할 확인

모든 관계 역할이 v5 목록 안에 있는지 확인한다(구조화 출력이라 보통 0이다).

In [4]:
# ── 역할 확인: 모든 관계 역할이 v5 목록 안에 있는지 ──
bad_role = sum(x['role'] not in ROLES_V5 for o in results.values() for x in o['relations'])
print(f'v5 목록에 없는 역할 {bad_role}개')
assert bad_role == 0

v5 목록에 없는 역할 0개


## 5. ① 근거 바로잡기

근거(`evidence`)는 기사 원문을 그대로 옮긴 것이어야 한다(공백 차이는 무시). 다르면 입력 글의 줄마다 근거와 길이가 비슷한 부분을 잘라 글자 일치율을 재고, 0.85 이상이면 그 원문 부분으로 바꾼다. 0.85 미만이면 그 근거와, 그 근거를 쓰는 관계·사건을 뺀다(사건은 다른 근거가 남아 있으면 남긴다).

In [5]:
# ── ① 근거 바로잡기: 원문과 다른 근거를 원문의 가장 비슷한 부분으로 바꾸거나 뺀다 ──
MATCH_MIN = 0.85


def best_match(evidence, text):
    """원문에서 근거와 가장 비슷한 부분과 그 일치율을 찾는다."""
    e = squash(evidence)
    # 근거를 6글자 조각으로 나눠, 조각이 하나라도 들어 있는 줄만 비교한다(빠르게)
    grams = {e[i:i + 6] for i in range(0, max(1, len(e) - 5), 3)}
    best = (0.0, '')
    for line in text.splitlines():
        flat = squash(line)
        if not flat or not any(g in flat for g in grams):
            continue
        words = line.split()
        # 줄이 근거와 길이가 비슷하면 줄 전체를, 길면 근거 길이(90~115%)만큼 단어를 잘라 후보로 쓴다
        spans = [line.strip()] if len(flat) <= len(e) * 1.15 else []
        if not spans:
            for i in range(len(words)):
                acc = ''
                for j in range(i, len(words)):
                    acc += words[j]
                    if len(acc) > len(e) * 1.15:
                        break
                    if len(acc) >= len(e) * 0.9:
                        spans.append(' '.join(words[i:j + 1]))
        for span in spans:
            ratio = difflib.SequenceMatcher(None, e, squash(span), autojunk=False).ratio()
            if ratio > best[0]:
                best = (ratio, span)
    return best


fixed = dropped = 0
for a, o in results.items():
    src = squash(TEXT[a])
    for ev in list(o['evidence']):
        # 공백만 다르면 원문 그대로인 것으로 본다
        if squash(ev['text']) in src:
            continue
        ratio, span = best_match(ev['text'], TEXT[a])
        if ratio >= MATCH_MIN:
            log(a, '①', 'evidence', ev['text'], span, f'원문과 다른 근거를 원문으로 바꿈(일치율 {ratio:.2f})')
            ev['text'] = span
            fixed += 1
            continue
        # 비슷한 부분이 없으면 근거를 빼고, 그 근거만 쓰던 관계·사건도 뺀다
        o['evidence'].remove(ev)
        log(a, '①', 'evidence', ev['text'], '', f'원문에서 찾을 수 없는 근거(일치율 {ratio:.2f})')
        dropped += 1
        for x in list(o['relations']):
            if x['evidence_id'] == ev['id']:
                o['relations'].remove(x)
                log(a, '①', 'relations', rel_label(o, x), '', '근거를 원문에서 찾을 수 없어 뺌')
        for x in list(o['events']):
            ids = x.get('evidence_ids') or [x.get('evidence_id')]
            if ev['id'] in ids:
                x['evidence_ids'] = [i for i in ids if i != ev['id']]
                if not x['evidence_ids']:
                    o['events'].remove(x)
                    log(a, '①', 'events', x['description_ko'], '', '근거를 원문에서 찾을 수 없어 뺌')
print(f'근거 바꿈 {fixed}개, 뺌 {dropped}개')

근거 바꿈 8개, 뺌 0개


## 6. ② 관계로 보지 않는 것 빼기

- 주체와 대상이 같은 개체인 관계.
- 증권사 투자의견을 적은 관계: v5 지시문은 뽑지 않게 하지만, 남은 경우를 거른다("기타" 관계 설명에 투자의견·목표주가·비중확대 등).
- 주체·대상·매개 제품이 개체 목록에 없는 관계.
- 근거 번호가 근거 목록에 없는 관계(v5 전체 추출에서 2건: 모델이 근거 문장을 목록에 적지 않음). 근거 없는 관계는 확인할 수 없어 뺀다. 사건은 없는 근거 번호만 빼고, 남는 근거가 없으면 사건을 뺀다.

In [6]:
# ── ② 관계로 보지 않는 것 빼기: 자기 자신, 증권사 투자의견, 없는 개체, 없는 근거 번호 ──
# 증권사 투자의견 표현(지시문이 막지만 "기타"로 남은 경우를 거른다)
RATING_RE = re.compile(r'투자\s*의견|목표\s*주가|비중\s*(확대|축소)|매수\s*의견|매도\s*의견|등급\s*(상향|하향)|시장수익률|outperform|overweight|underweight', re.I)

removed = {'자기 자신': 0, '투자의견': 0, '없는 개체': 0, '없는 근거 번호': 0, '사건: 근거 없음': 0}
for a, o in results.items():
    ids = {e['id'] for e in o['entities']}
    evs = {e['id'] for e in o['evidence']}
    for x in list(o['relations']):
        why = None
        if x['subject'] == x['object']:
            why = '자기 자신'
        elif x['role'] == '기타' and RATING_RE.search(x['role_note']):
            why = '투자의견'
        elif x['subject'] not in ids or x['object'] not in ids or (x['product'] and x['product'] not in ids):
            why = '없는 개체'
        elif x['evidence_id'] not in evs:
            why = '없는 근거 번호'
        if why:
            o['relations'].remove(x)
            log(a, '②', 'relations', f"{rel_label(o, x)} ({x['role_note']})", '', why)
            removed[why] += 1
    # 사건은 없는 근거 번호만 빼고, 남는 근거가 없으면 사건을 뺀다
    for x in list(o['events']):
        keep = [i for i in x['evidence_ids'] if i in evs]
        if len(keep) < len(x['evidence_ids']):
            log(a, '②', 'events', x['description_ko'][:80], ', '.join(keep), '없는 근거 번호 뺌')
            x['evidence_ids'] = keep
        if not keep:
            o['events'].remove(x)
            removed['사건: 근거 없음'] += 1
print('뺀 관계', removed)

뺀 관계 {'자기 자신': 0, '투자의견': 0, '없는 개체': 0, '없는 근거 번호': 2, '사건: 근거 없음': 0}


## 7. ③ 겹치는 관계 합치기

- 같은 주체·역할·대상·매개 제품의 관계가 여러 번 있으면 처음 것만 남긴다.
- 공급(A → B, 제품 P)이 있으면 같은 거래를 사는 쪽에서 다시 적은 구매·사용(B → P 또는 B → A)은 뺀다(v5 지시문 규칙).

In [7]:
# ── ③ 겹치는 관계 빼기: 같은 관계 중복, 공급과 같은 거래인 구매·사용 ──
dup = buy = 0
for a, o in results.items():
    seen = set()
    for x in list(o['relations']):
        # 주체·역할·대상·제품이 같으면 같은 관계
        key = (x['subject'], x['role'], x['object'], x['product'])
        if key in seen:
            o['relations'].remove(x)
            log(a, '③', 'relations', rel_label(o, x), '', '같은 관계 중복')
            dup += 1
        seen.add(key)
    # A → 공급 → B(제품 P)가 있으면, B → 구매·사용 → A 또는 P는 같은 거래라 뺀다
    supplies = [(x['subject'], x['object'], x['product']) for x in o['relations'] if x['role'] == '공급']
    for x in list(o['relations']):
        if x['role'] != '구매·사용':
            continue
        if any(x['subject'] == b and (x['object'] == s or (p and x['object'] == p)) for s, b, p in supplies):
            o['relations'].remove(x)
            log(a, '③', 'relations', rel_label(o, x), '', '공급 관계와 같은 거래')
            buy += 1
print(f'중복 뺌 {dup}개, 공급과 겹친 구매·사용 뺌 {buy}개')

중복 뺌 47개, 공급과 겹친 구매·사용 뺌 0개


## 8. ④ 너무 일반적인 검색어 빼기

수많은 기사에 나오는 일반어는 검색어로 쓰면 관계없는 기사가 너무 많이 걸린다. 아래 목록에 있는 검색어만 뺀다(대소문자 무시). 개체 자체는 남긴다. 무엇이 일반어인지는 제품 사전 단계에서 다시 본다.

In [8]:
# ── ④ 너무 일반적인 검색어 빼기: 개체는 남기고 검색어에서만 뺀다 ──
# 이 말로 검색하면 관련 없는 기사가 너무 많이 걸린다
GENERIC_TERMS = {'ai', 'artificial intelligence', 'oil', 'crude', 'gas', 'electricity', 'power', 'energy', 'data centers',
                 'data center', 'inflation', 'stocks', 'tariffs', 'evs', 'electric vehicles', 'batteries', 'chips',
                 'semiconductors', 'technology', 'software', 'cloud', 'steel', 'shipping', 'insurance', 'banks'}
removed_terms = 0
for a, o in results.items():
    for e in o['entities']:
        keep = [t for t in e['search_terms'] if t.strip().lower() not in GENERIC_TERMS]
        if len(keep) != len(e['search_terms']):
            log(a, '④', 'entities', f"{e['name_en']}: {e['search_terms']}", keep, '일반적인 검색어')
            removed_terms += len(e['search_terms']) - len(keep)
            e['search_terms'] = keep
print('뺀 검색어', removed_terms)

뺀 검색어 11


## 9. 저장

- 아무 관계·사건도 쓰지 않게 된 근거는 근거 목록에서 지운다(기록하지 않는다).
- 기사마다 메타 정보를 덧붙인다: 매체, 제목, 주소, 발행 시각, 입력 종류(본문/요약), 지시문 버전, 실행 수단, 원본 결과 파일.
- 파일 내용이 바뀌었을 때만 쓴다(iCloud 동기화 사본 방지).

In [9]:
# ── 저장: 안 쓰인 근거를 지우고 기사 정보를 붙여 최종본과 변경 기록을 쓴다 ──
def write_if_changed(path, content):
    """내용이 바뀌었을 때만 파일을 쓴다(iCloud 동기화 사본이 생기지 않게)."""
    if path.exists() and path.read_text(encoding='utf-8') == content:
        return False
    path.write_text(content, encoding='utf-8')
    return True


final = {'model': MODEL, 'roles': ROLES_V5, 'articles': {}}
for a, o in results.items():
    # 관계·사건이 가리키지 않는 근거는 최종본에서 뺀다(정보 손실이 없어 기록하지 않는다)
    used = {x['evidence_id'] for x in o['relations']} | {i for x in o['events'] for i in (x.get('evidence_ids') or [x.get('evidence_id')])}
    o['evidence'] = [e for e in o['evidence'] if e['id'] in used]
    r = art.loc[a]
    final['articles'][a] = {'source': r['source'], 'title': r['title'], 'url': r['url'], 'published_utc': r['published_utc'],
                            'input': r['use'], **meta[a], **o}

change_df = pd.DataFrame(changes, columns=['article_id', 'title', 'rule', 'section', 'before', 'after', 'why'])
w1 = write_if_changed(FINAL / 'news_final.json', json.dumps(final, ensure_ascii=False, indent=1))
w2 = write_if_changed(FINAL / 'changes.csv', '﻿' + change_df.to_csv(index=False))
print(f'최종본 {"저장" if w1 else "바뀐 것 없음"} ({(FINAL / "news_final.json").stat().st_size / 1e6:.1f}MB), '
      f'변경 기록 {"저장" if w2 else "바뀐 것 없음"} ({len(change_df)}줄)')

최종본 바뀐 것 없음 (14.7MB), 변경 기록 저장 (64줄)


## 10. 결과 확인

1. 번호별 변경 건수
2. 항목 수: 원본 → 최종본
3. 최종본 점검: 모든 근거가 원문에 있는지, 관계·사건이 가리키는 근거·개체가 모두 있는지, 역할이 v5 목록 안에 있는지
4. 관계 역할 분포

In [10]:
# ── 결과 확인 1·2: 규칙별 변경 건수, 정리 전후 항목 수 ──
print('1) 번호별 변경 건수')
display(change_df.groupby(['rule', 'why']).size().rename('건수').reset_index())

print('2) 항목 수')
before = pd.DataFrame(raw_counts).T.sum()
after = pd.Series({k: sum(len(o[k]) for o in results.values()) for k in ('entities', 'relations', 'events', 'evidence')})
display(pd.DataFrame({'원본': before, '최종본': after, '뺀 것': before - after}))

1) 번호별 변경 건수


,rule,why,건수
0,①,원문과 다른 근거를 원문으로 바꿈(일치율 0.98),2
1,①,원문과 다른 근거를 원문으로 바꿈(일치율 0.99),4
2,①,원문과 다른 근거를 원문으로 바꿈(일치율 1.00),2
3,②,없는 근거 번호,2
4,③,같은 관계 중복,47
5,④,일반적인 검색어,7


2) 항목 수


,원본,최종본,뺀 것
entities,19029,19029,0
relations,7949,7900,49
events,4262,4262,0
evidence,12286,10945,1341


In [11]:
# ── 결과 확인 3·4: 최종본 점검(어긋나면 멈춤), 관계 역할 분포 ──
print('3) 최종본 점검')
bad_ev = bad_ref = bad_id = bad_role = 0
for a, o in final['articles'].items():
    src = squash(TEXT[a])
    ev = {e['id'] for e in o['evidence']}
    ids = {e['id'] for e in o['entities']}
    bad_ev += sum(squash(e['text']) not in src for e in o['evidence'])
    bad_id += sum(x['evidence_id'] not in ev for x in o['relations'])
    bad_id += sum(i not in ev for x in o['events'] for i in (x.get('evidence_ids') or [x.get('evidence_id')]))
    bad_ref += sum(x['subject'] not in ids or x['object'] not in ids for x in o['relations'])
    bad_role += sum(x['role'] not in ROLES_V5 for x in o['relations'])
print(f'원문에 없는 근거 {bad_ev}, 없는 근거 번호 {bad_id}, 없는 개체 {bad_ref}, v5 목록 밖 역할 {bad_role}')
assert bad_ev == bad_ref == bad_id == bad_role == 0

print('4) 관계 역할 분포')
roles = pd.Series([x['role'] for o in final['articles'].values() for x in o['relations']]).value_counts()
display(pd.DataFrame({'관계 수': roles, '비율 %': (roles / roles.sum() * 100).round(1)}))

3) 최종본 점검


원문에 없는 근거 0, 없는 근거 번호 0, 없는 개체 0, v5 목록 밖 역할 0
4) 관계 역할 분포


,관계 수,비율 %
생산,2279,28.8
인수·투자,868,11.0
합작·제휴,810,10.3
시설 건설·운영,735,9.3
규제·인허가,727,9.2
구매·사용,575,7.3
공급,477,6.0
기타,241,3.1
정책 입장,231,2.9
수주·발주,200,2.5
